In [1]:
import pandas as pd

weather_df = pd.read_csv("Dataset3_Weather.csv")
df = pd.read_csv("Dataset1_Cleaned_AGMARKNET.csv")
market_df = pd.read_csv("Dataset2_Cleaned_Market_Locations.csv")

In [3]:
print(df.shape)
print(market_df.shape)
print(weather_df.shape)

(9049, 10)
(1391, 5)
(1388, 8)


In [4]:
missing_weather = market_df[
    ~market_df["market"].isin(weather_df["market"])
]

print("Missing Markets :", len(missing_weather))

missing_weather

Missing Markets : 3


,state,district,market,latitude,longitude
76,Tamil Nadu,Dharmapuri,Karimangalam(Uzhavar Sandhai),12.304705,78.208163
623,Madhya Pradesh,Dhar,Rajgarh APMC,22.677601,74.947502
1239,Tamil Nadu,Coimbatore,Thiruppur APMC,10.793908,77.269322


In [5]:
import requests
import pandas as pd
import time

new_weather = []

for _, row in missing_weather.iterrows():

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={row['latitude']}"
        f"&longitude={row['longitude']}"
        f"&daily=temperature_2m_max,"
        f"temperature_2m_min,"
        f"precipitation_sum,"
        f"weather_code,"
        f"wind_speed_10m_max"
        f"&timezone=auto"
    )

    response = requests.get(url)

    data = response.json()

    daily = data["daily"]

    new_weather.append({

        "market": row["market"],

        "latitude": row["latitude"],

        "longitude": row["longitude"],

        "temperature": (
            daily["temperature_2m_max"][0] +
            daily["temperature_2m_min"][0]
        ) / 2,

        "humidity": None,

        "rainfall": daily["precipitation_sum"][0],

        "wind_speed": daily["wind_speed_10m_max"][0],

        "weather_code": daily["weather_code"][0]

    })

    time.sleep(1)

new_weather = pd.DataFrame(new_weather)

new_weather

,market,latitude,longitude,temperature,humidity,rainfall,wind_speed,weather_code
0,Karimangalam(Uzhavar Sandhai),12.304705,78.208163,29.85,None,0.0,16.7,3
1,Rajgarh APMC,22.677601,74.947502,27.25,None,0.6,28.1,51
2,Thiruppur APMC,10.793908,77.269322,29.25,None,0.0,34.1,3


In [6]:
weather_df = pd.concat(
    [weather_df, new_weather],
    ignore_index=True
)

In [7]:
print(weather_df.shape)

(1391, 8)


In [8]:
print(weather_df["market"].duplicated().sum())

10


In [12]:
weather_df.duplicated(
    subset=["market" ,"latitude", "longitude"]
).sum()

0

In [13]:
weather_df.to_csv(
    "Dataset3_Cleaned_Weather.csv",
    index=False
)

print("Dataset 3 Saved Successfully")

Dataset 3 Saved Successfully
